# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and analyzing the FAIR^2 mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://github.com/mlcommons/croissant) and provided via a schema URL.

* **Title**: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
* **Citation**: Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.
* **Schema URL**: `https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`
* **License**: [Open Data Commons BY 1.0](https://opendatacommons.org/licenses/by/1-0/)


In [ ]:
# Install mlcroissant (if not already installed in environment)
!pip install --quiet mlcroissant

## 1. Data Loading

We will load the dataset metadata, fetch available record sets, and print a summary using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Print dataset name & description
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Explore available **Record Sets** in this Croissant dataset. Each record set, field, and column is referenced using its unique `@id` as defined in the schema. 

Let's list each record set's `@id`, name, and their field `@id`s.

In [ ]:
# List record sets and their fields by @id
record_sets = list(dataset.metadata.record_set)
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    if 'name' in rs:
        print(f"  Name: {rs['name']}")
    if 'description' in rs:
        print(f"  Description: {rs['description']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        # Support for a single field as dict
        fields = [fields]
    print("  Fields (@id):")
    for field in fields:
        print(f"    - {field['@id']}")
    print()

Below is an example of how to iterate through the records for a specific record set using its `@id`.



In [ ]:
# Example: Print sample records from the main protein record set
# Change the @id if you wish to explore another record set
# To find the right record set @id, check the cell above!

main_record_set_id = None
for rs in record_sets:
    if 'protein' in rs.get('name','').lower():
        main_record_set_id = rs['@id']
        break
if not main_record_set_id:
    # Fallback: use first
    main_record_set_id = record_sets[0]['@id']

print(f"Sample 2 records from record set '{main_record_set_id}':\n")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i == 1:
        break

## 3. Data Extraction

Let's extract all data from the main record set (`@id`: shown in the overview above) into a pandas DataFrame for data analysis. 

All data manipulations reference record set and field columns by their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded {len(df)} records from record set @id: {rec_id}")

# Display column names for main record set
print(f"\nMain record set columns (@id):\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's conduct common EDA steps. All field/column references are by `@id`.

* We'll identify and use one numeric field to filter values and normalize it.
* We'll attempt to group by a main categorical field (`description`, `accession`, or similar) if present.


In [ ]:
# Choose a numeric field for filtering & normalization.
# You may check available columns above. For this dataset, 'cr:coverage_percent' or 'cr:mw_kda' (molecular weight) are plausible numeric fields.
numeric_field_id = None
possible_numeric_fields = ['cr:coverage_percent', 'cr:mw_kda', 'cr:peptide_count', 'cr:psm_count', 'cr:abundance__sample1', 'cr:abundance__sample2', 'cr:abundance__sample3']

main_df = dataframes[main_record_set_id]
for fid in possible_numeric_fields:
    if fid in main_df.columns:
        numeric_field_id = fid
        break

if not numeric_field_id:
    # Fallback to first numeric field
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

print(f"Using numeric field: {numeric_field_id}\n")

# Filter for top values (e.g., coverage percent > 10)
threshold = 10
if numeric_field_id:
    filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:\n")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"Normalized {numeric_field_id} (mean 0, std 1):\n")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No numeric field found for EDA.")

# Try grouping by a major categorical field, e.g., cr:accession or cr:description
group_field = None
for candidate in ['cr:description', 'cr:accession', 'cr:gene_symbol', 'cr:protein_name']:
    if candidate in main_df.columns:
        group_field = candidate
        break

if group_field and numeric_field_id and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
    print(f"\nTop groups for {group_field} by mean {numeric_field_id} (> threshold):\n")
    display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the filtered numeric variable and display top proteins (if available), using only record/field `@id`s for reference.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(filtered_df[numeric_field_id], errors='coerce'), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Show top N by value
    if group_field:
        top_n = filtered_df[[group_field, numeric_field_id]].sort_values(numeric_field_id, ascending=False).head(10)
        print(f"Top 10 {group_field} entries by {numeric_field_id}:")
        display(top_n)

else:
    print("No numeric field data to visualize.")

## 6. Conclusion

* This notebook demonstrated how to load and explore a Croissant-standard dataset using the `mlcroissant` library, referencing all dataset elements by their `@id`.
* We performed filtering and normalization on the selected numeric field (referenced by its `@id`).
* We grouped protein records by a categorical field (`@id`) and visualized the data, showing how Croissant schema enables reliable, schema-based data handling.

You may modify field and record set `@id` selections in the notebook for deeper exploration of other components of this FAIR^2 dataset.
